In [1]:
from helpers.persistence import save_var, load_var
# from helpers.tabular_data_loader import data_loaders, get_label_idx
from helpers.progress_bar import ProgressBar

In [2]:
import numpy as np
from tqdm import tqdm
import scipy

In [3]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, label_binarize
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import pairwise_distances, pairwise_distances_chunked

import pandas as pd

In [4]:
from helpers.openml_data_v2 import openml_cc18_list, hard_list, get_data1, get_data_raw
from helpers.openml_data import tabular_id_list

from time import time

In [5]:

# import os
# os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2' 

import tensorflow as tf
import logging
logging.getLogger('tensorflow').setLevel(logging.ERROR)

/home/cida-lab-2/miniconda3/envs/tabnet/lib/python3.6/site-packages/tensorflow/python/framework/dtypes.py:523: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_qint8 = np.dtype([("qint8", np.int8, 1)])
/home/cida-lab-2/miniconda3/envs/tabnet/lib/python3.6/site-packages/tensorflow/python/framework/dtypes.py:524: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_quint8 = np.dtype([("quint8", np.uint8, 1)])
/home/cida-lab-2/miniconda3/envs/tabnet/lib/python3.6/site-packages/tensorflow/python/framework/dtypes.py:525: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_qint16 = np.dtype([("qint16", np.int16, 1)])
/home/cida-lab-2/miniconda3/env

In [6]:

def get_col_types(df0, y, cat_mask):
    
    df = df0.copy(deep=0)
    df.columns = [str(i) for i in df.columns]
    
    # display(df)
    
    rows = []
    
    for c, is_cat in zip(df.columns, cat_mask):
        if is_cat:
            # make sure original data have string values in cat columns
            rows.append([c, df[c].unique().shape[0], 'str'])
        else:
            rows.append([c, df[c].unique().shape[0], df[c].dtype.name])
            
    # print(rows)
    
    df = pd.DataFrame(rows, columns=['column_name', 'n_uniques', 'col_type'])
    # df['col_type'] = df.apply(lambda x: 'bool' if x.n_uniques==2 and x.type=='str' else x.type, axis=1)
    df['col_type'] = df['col_type'].str.lower()
    
    # display(df)
    
    LABEL_COLUMN = "Target"

    BOOL_COLUMNS = [row.column_name for i, row in df.iterrows() if 'bool' in row.col_type]

    INT_COLUMNS = [row.column_name for i, row in df.iterrows() if 'int' in row.col_type]

    STR_COLUMNS = [row.column_name for i, row in df.iterrows() if 'str' in row.col_type]
    STR_NUNIQUESS = [row.n_uniques for i, row in df.iterrows() if 'str' in row.col_type] 
    
    # print('STR_COLUMNS', STR_COLUMNS)

    FLOAT_COLUMNS = [row.column_name for i, row in df.iterrows() if 'float' in row.col_type]

    DEFAULTS = ([[0] for col in INT_COLUMNS] + [[""] for col in BOOL_COLUMNS] +
                [[0.0] for col in FLOAT_COLUMNS] + [[""] for col in STR_COLUMNS] +
                [[-1]])
    
    # print(DEFAULTS)

    FEATURE_COLUMNS = (
        INT_COLUMNS + BOOL_COLUMNS + FLOAT_COLUMNS + STR_COLUMNS)
    ALL_COLUMNS = FEATURE_COLUMNS + [LABEL_COLUMN]
    
    # print(FEATURE_COLUMNS)
    
    
    def get_columns():
        """Get the representations for all input columns."""

        columns = []
        if FLOAT_COLUMNS:
            columns += [tf.feature_column.numeric_column(ci) for ci in FLOAT_COLUMNS]
        if INT_COLUMNS:
            columns += [tf.feature_column.numeric_column(ci) for ci in INT_COLUMNS]
        if STR_COLUMNS:
            # pylint: disable=g-complex-comprehension
            columns += [
                tf.feature_column.embedding_column(
                    tf.feature_column.categorical_column_with_hash_bucket(
                        ci, hash_bucket_size=int(3 * num)),
                    dimension=1) for ci, num in zip(STR_COLUMNS, STR_NUNIQUESS)
            ]
        if BOOL_COLUMNS:
            # pylint: disable=g-complex-comprehension
            columns += [
                tf.feature_column.embedding_column(
                    tf.feature_column.categorical_column_with_hash_bucket(
                        ci, hash_bucket_size=3),
                    dimension=1) for ci in BOOL_COLUMNS
            ]
        return columns


    def parse_csv(value_column):
        """Parses a CSV file based on the provided column types."""
        columns = tf.decode_csv(value_column, record_defaults=DEFAULTS)
        # print(columns)
        features = dict(zip(ALL_COLUMNS, columns))
        label = features.pop(LABEL_COLUMN)
        classes = tf.cast(label, tf.int32) - 1
        return features, classes


    def input_fn(data_file,
                 num_epochs,
                 shuffle,
                 batch_size,
                 n_buffer=50,
                 n_parallel=4):
        """Function to read the input file and return the dataset.

        Args:
          data_file: Name of the file.
          num_epochs: Number of epochs.
          shuffle: Whether to shuffle the data.
          batch_size: Batch size.
          n_buffer: Buffer size.
          n_parallel: Number of cores for multi-core processing option.

        Returns:
          The Tensorflow dataset.
        """

        # Extract lines from input files using the Dataset API.
        dataset = tf.data.TextLineDataset(data_file)
        # print(dataset)

        if shuffle:
            dataset = dataset.shuffle(buffer_size=n_buffer)

        dataset = dataset.map(parse_csv, num_parallel_calls=n_parallel)
        
        # print(data_file, num_epochs, batch_size, len(dataset))
        
        # Repeat after shuffling, to prevent separate epochs from blending together.
        dataset = dataset.repeat(num_epochs)
        dataset = dataset.batch(batch_size)
        return dataset
    
    return FEATURE_COLUMNS, get_columns(), input_fn

In [7]:
573*3 / 128

13.4296875

In [8]:
# n_samples = 900
# n_epochs = 2
# batch_size = 150
# n_steps = int(np.ceil(n_samples*n_epochs/batch_size))
# print('n steps = ', n_steps)

# actual_dataset = tf.data.Dataset.from_tensor_slices(list(range(1,n_samples+1)))
# dataset = actual_dataset.shuffle(buffer_size=n_samples).repeat(n_epochs).batch(batch_size)

# dataset_iter = dataset.make_initializable_iterator()
# x = dataset_iter.get_next()

# with tf.Session() as sess:
#     sess.run(dataset_iter.initializer)
    
#     for i in range(n_steps):
#         samples = sess.run(x)
#         print(i, len(samples))
#         # continue
#     # print(i)

In [9]:
import tabnet.experiment_covertype as tabnet
run_tabnet = tabnet.main 
tabnet.BATCH_SIZE = 128
tabnet.MAX_EPOCH = 1000
tabnet.PATIENCE = -1

In [10]:
def reset_label_order(labels):
    labels = pd.Series(labels)
    unique_labels = list(labels.unique())
    unique_labels.sort()
    
    reordered_labels = [unique_labels.index(label) for label in labels]
    
    return np.array(reordered_labels)

In [11]:
# df, labels, cats = get_data1(1510)
# np.ceil(int(df.shape[0]*.7)*2/128)
# np.ceil(int(df.shape[0]*.2))

# for c, is_cat in zip(df.columns, cats):
#     if ~is_cat: continue
#     df[c] = df[c].apply(lambda x: f'{c}_value_{x}')
    
    
# # df1 = pd.DataFrame(df.values)#.dtypes
# df.values

In [12]:
def do_kfold_cv(dataset_id, pbar = False, cache_key = 'tabnet-data'):
    df, labels, cats = get_data1(dataset_id)
    df.columns = [f'col_{i}' for i,x in enumerate(df.columns)]
    
    for c, is_cat in zip(df.columns, cats):
        if ~is_cat: continue
        df[c] = df[c].apply(lambda x: f'{c}_{x}')
    
    X = df.values
    # print('X[0,:]', X[0,:])
    
    # tabnet preprocessing substracts from labels because example (covtype) labels were 1-7 not 0-6 
    # and -1 is the default (in preprocessor) if label is missing; see data_helper*.py#L97
    # labels in dermatology dataset starts from 1-6 not 0-5
    y = reset_label_order(labels) + 1
    
    num_samples = X.shape[0]
    num_features = X.shape[1]
    # print(f'num features: {num_features}')
    num_classes = np.unique(y).shape[0]
    
    
    # start tabnet preprocessing
    
    feature_columns, columns, input_fn = get_col_types(df, y, cats)
    # print('feature_columns', feature_columns)
    # print('columns:', len(columns))
    
    # df = pd.DataFrame(X)
    # df.columns = [str(i) for i in df.columns]
    df_reordered = df[feature_columns]
    X = df_reordered.values #column reordered
    # print('reordered X[0,:]', X[0,:])
    
    # end tabnet preprocessing
    
    idx = list(range(X.shape[0]))
    idx_splits = [train_test_split(idx, test_size=0.2, random_state=i, stratify = y) for i in range(30)]
    
    val_scores = []
    test_scores = []
    y_preds = []
    y_trues = []
    time_taken_list = []
    
    pbar and pbar.add_prefix(f'starting 30-fold bootstrap on {cache_key}')

    for fold_index, (train_index, test_index) in enumerate(idx_splits):
        pbar and pbar.edit_last_prefix(f'{cache_key} | Test: {fold_index+1} / {len(idx_splits)} |')
        
        
        cache_items = cache_key.split('_')
        cache_key = '_'.join(cache_items[:1])
        cache_prefix = f'{cache_key}_test-{fold_index}'
        cache_folder = './saved_vars/tabnet_folds'
        csv_paths = [f'{cache_folder}/{cache_prefix}-{fold_name}.csv' for fold_name in ['train','val','test']]
        fold_sizes_path = f'{cache_folder}/{cache_prefix}_table_sizes.pkl'
        
        try:
            raise Exception('Doing again') # recreate fold csv anyway
            pbar and pbar.set_description(f'loading {cache_prefix}')
            for csv_path in csv_paths:
                df_fold = pd.read_csv(csv_path, header=None)
                
            fold_sizes = load_var(fold_sizes_path) or [0,0,0]
            
            if fold_sizes[0]==0:
                raise Exception('Fold sizes is 0')
            
        except Exception as e:
            # raise e
            pbar and pbar.set_description(f'Cant load {cache_prefix}, generating')
            
            X_train, X_test = X[train_index], X[test_index]
            y_train, y_test = y[train_index], y[test_index]

            X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=1/8, random_state=42, stratify = y_train)
            
            # print('n_train_samples', X_train.shape[0])
            tabnet.MAX_STEPS = int(np.ceil(X_train.shape[0] * tabnet.MAX_EPOCH / tabnet.BATCH_SIZE))
            # tabnet.MAX_STEPS = 1000
            
            fold_sizes = []

            for fold_x, fold_y, csv_path in zip([X_train, X_val, X_test] , [y_train, y_val, y_test], csv_paths):
                # print('fold X[0,:]', fold_x[0,:])
                df_fold = pd.DataFrame(fold_x)
                df_fold['Target'] = fold_y
                fold_sizes.append(df_fold.shape[0])
                df_fold.to_csv(csv_path, header=False, index=False)
                
            save_var(fold_sizes, fold_sizes_path)
            
        
        # pbar and print('pbar casts to True')
        tf.reset_default_graph()
        
        # print(csv_paths)
        # print(fold_sizes)
        
        start_time = time()
        
        val_score, test_score, y_pred = run_tabnet(csv_paths[0], csv_paths[1], csv_paths[2], 
                                           fold_sizes[0], fold_sizes[1], fold_sizes[2], #
                                           num_features, 
                                           num_classes,
                                           columns, 
                                           feature_columns, 
                                           input_fn,
                                           model_name=cache_prefix, pbar=pbar)
        
        time_taken = time() - start_time
        
        # print('returned:', val_score, test_score)
        
        y_test = y[test_index]
        # since we added one for the preprocessor, we now undo that
        y_test = y_test - 1
        score = f1_score(y_test, y_pred, average='weighted')
        
        # print('f1', y_pred, y_test, score)
        
        val_scores.append(val_score)
        test_scores.append(score)
        y_preds.append(y_pred)
        y_trues.append(y_test)
        time_taken_list.append(time_taken)
        
    pbar and pbar.clear_prefix()
        
    return val_scores, test_scores, y_preds, y_trues, time_taken_list
        
        

In [13]:
save_path, export_path = './saved_vars/test-time-tabnet.pkl', './exports/test-time-tabnet.csv'
dataset_results = load_var(save_path) or {}
# dataset_results = {}

Could not load ./saved_vars/test-time-tabnet.pkl because [Errno 2] No such file or directory: './saved_vars/test-time-tabnet.pkl'


In [14]:
names=[1510]
names = [   23,   458,   469,  1049,  1050,  1063,  1067,  1068,  1464,
        1475,  1485,  1487,  1494,  1497,  1510,  4134,  4538, 40701,
       40975, 40982];
# names = names[:10]
names = names [-10:-5]
names = [1510]
pbar = ProgressBar(names)
# pbar = False

for dataset_name in pbar:

    key = f"{dataset_name}_tabnet"
    pbar.set_description(f'Doing {key}')
    pbar and pbar.clear_prefix()

    if key in dataset_results.keys():
        print(key, 'already done')
        continue

    try:
        results = do_kfold_cv(dataset_name, pbar=pbar, cache_key = f'{dataset_name}')
        
        print(f'Total time for 30 times: {np.sum(results[-1])}; mean = {np.mean(results[-1])}')
                                          
        dataset_results[key] = results
        save_var(dataset_results, save_path)
        
    except Exception as e:
        print(f'Could not do {dataset_name}')
        print(e)
        raise e
        
    


1510 | Test: 30 / 30 | Step :3110/3110, Training Loss = 0.0076, Val Accuracy: 0.9298 (best: 0.9825): 100%|██████████| 1/1 [2:53:15<00:00, 10395.12s/it]

Total time for 30 times: 10394.201371908188; mean = 346.4733790636063


In [15]:
rows = []
cols = ['dataset', 'model', 'fold', 'val_score','test_score']

for k,v in dataset_results.items():
    k = k.split('_')
    
    for i in range(len(v[0])):
        rows.append([k[0], k[1], i, v[0][i], v[1][i]])
                     
df = pd.DataFrame(rows, columns=cols)

In [16]:
df.dataset.unique()

array(['1510'], dtype=object)

In [17]:
df[['dataset','model','fold','test_score']].to_csv(export_path, index=0)

In [18]:
df.groupby(['dataset','model']).test_score.agg('mean')

dataset  model 
1510     tabnet    0.953267
Name: test_score, dtype: float64